# Week 4 · Day 3 — Pandas: Series, DataFrames & Real Data

Pandas is the library data scientists live in. If NumPy gives us fast arrays of numbers, Pandas gives us **labeled tables** — rows and columns with names, mixed data types, and missing values handled gracefully. It is built on top of NumPy, so everything we learned still applies underneath.

**Today we cover:**
1. Series — a single labeled column
2. DataFrame — a full table
3. Loading a real dataset and inspecting it
4. Selecting rows and columns (`loc`, `iloc`, conditional filtering)
5. `describe()` — every statistic from this week in one line
6. Data types: how to classify what each column actually *is*
7. Everyday DataFrame operations

In [ ]:
import numpy as np
import pandas as pd

---
## 1. The Series — a NumPy array with names

A **Series** is a one-dimensional array where every value has a **label** (called the *index*). Think of it as a single column of a spreadsheet, or a dictionary that behaves like an array.

In [ ]:
# From a plain list — Pandas auto-numbers the index 0,1,2...
temps = pd.Series([31, 35, 29, 33])
temps

In [ ]:
# From a list with custom labels — now the index is meaningful
temps = pd.Series([31, 35, 29, 33], index=["Mon", "Tue", "Wed", "Thu"])
temps

In [ ]:
# Look up by label, just like a dictionary
temps["Tue"]

In [ ]:
# From a dictionary — keys become the index automatically
population = pd.Series({"Karachi": 16.0, "Lahore": 11.1, "Islamabad": 1.1})
population

A Series carries the NumPy skills forward: the same math and the same boolean filtering work, but now the labels come along for the ride.

In [ ]:
print(temps.mean())          # NumPy-style aggregation
print(temps[temps > 30])     # boolean filtering, labels preserved

---
## 2. The DataFrame — a full table

A **DataFrame** is a collection of Series sharing the same index — in other words, a spreadsheet: rows and named columns. This is the object we will use for the rest of the course.

In [ ]:
# Build one from a dictionary: each key is a column, each list is that column's values
df = pd.DataFrame({
    "name":   ["Ali", "Sara", "Bilal", "Ayesha"],
    "age":    [23, 35, 29, 41],
    "city":   ["Lahore", "Karachi", "Lahore", "Islamabad"],
    "salary": [55000, 82000, 61000, 95000],
})
df

In [ ]:
# A single column is a Series
df["age"]

In [ ]:
# Multiple columns → pass a list of names → get a smaller DataFrame
df[["name", "salary"]]

---
## 3. Loading a real dataset

In practice data comes from files, most often CSV (comma-separated values). `pd.read_csv()` reads one into a DataFrame. Here we build a small Titanic-style dataset in code so the notebook is self-contained, but in the lab this single line does the job:

```python
df = pd.read_csv("titanic.csv")
```

In [ ]:
# A compact Titanic-style dataset (real columns, deliberately includes missing ages)
data = {
    "Survived": [0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1],
    "Pclass":   [3, 1, 3, 1, 3, 3, 1, 3, 3, 2, 3, 1],
    "Sex":      ["male","female","female","female","male","male",
                 "male","female","female","female","male","female"],
    "Age":      [22, 38, 26, 35, np.nan, np.nan, 54, 27, 14, 4, 20, 58],
    "Fare":     [7.25, 71.28, 7.92, 53.10, 8.05, 8.46,
                 51.86, 11.13, 30.07, 16.70, 7.85, 26.55],
    "Embarked": ["S","C","S","S","S","Q","S","S","S","S","S","C"],
}
df = pd.DataFrame(data)
df

### The first-contact ritual

Before any analysis, look at the shape and structure of the data. Run these five on *every* dataset you ever open — they answer "how big is it, what are the columns, and what type is each one?" 

In [ ]:
df.head()        # first 5 rows — a quick glance

In [ ]:
df.shape         # (rows, columns)

In [ ]:
df.info()        # column names, non-null counts, and data types — note Age has fewer non-nulls

In [ ]:
df.dtypes        # the type of each column at a glance

---
## 4. Selecting rows and columns

Selecting columns is easy (`df["col"]`). Selecting **rows** uses two tools, and the difference between them matters:

- **`loc`** — select by **label** (the index name)
- **`iloc`** — select by **integer position** (0, 1, 2, ...)

In [ ]:
df.loc[0]        # the row with index LABEL 0

In [ ]:
df.iloc[0]       # the row in POSITION 0 — same here because our index is 0,1,2...

In [ ]:
# loc can take row + column: df.loc[rows, columns]
df.loc[0:3, ["Sex", "Age"]]     # rows labelled 0..3, only Sex and Age

In [ ]:
# iloc uses positions and, like Python slicing, excludes the end
df.iloc[0:3, 1:4]               # first 3 rows, columns in positions 1,2,3

### Conditional selection — the boolean mask, now on a table

This is the same boolean filtering from NumPy: a comparison builds a True/False mask, and the DataFrame keeps only the `True` rows.

In [ ]:
# Everyone older than 30
df[df["Age"] > 30]

In [ ]:
# Combine conditions with &  |  — and wrap EACH condition in parentheses
df[(df["Age"] > 30) & (df["Pclass"] == 1)]     # over 30 AND first class

**Practical example — a real question, answered in one line.**
*"Who paid more than the average fare?"* The mask can contain a statistic.

In [ ]:
df[df["Fare"] > df["Fare"].mean()][["Sex", "Pclass", "Fare"]]

### Setting and resetting the index

Any column can become the index (the row labels). This is useful when a column is a natural identifier.

In [ ]:
df.set_index("Sex").head()      # Sex becomes the row label (temporary — returns a new frame)

---
## 5. `describe()` — the whole week in one method

Everything computed by hand on Days 1 and 2 — mean, standard deviation, min, max, and the quartiles — is produced for every numeric column by a single call.

In [ ]:
df.describe()

Reading the output row by row:

| Row | Meaning | Where we met it |
|---|---|---|
| **count** | how many non-missing values | (note Age < 12: missing data) |
| **mean** | the average | Day 1 |
| **std** | standard deviation — the spread | Day 2 |
| **min / max** | smallest / largest | Day 1 |
| **25% / 50% / 75%** | the quartiles (50% is the median) | Day 2 |

One method summarizes an entire column. `.describe()` is usually the second thing you run on any dataset, right after `.info()`.

---
## 6. Classifying data types

Before modelling, you must know what *kind* of thing each column is — it decides which statistics and which plots make sense.

**Structured vs unstructured**
- *Structured* — fits neatly in rows and columns (everything in this table).
- *Unstructured* — free text, images, audio (e.g. a passenger's full name).

**Quantitative (numbers you can do math on)**
- *Continuous* — any value in a range (Age, Fare).
- *Discrete* — whole counts (number of siblings aboard).

**Qualitative (categories, not magnitudes)**
- *Nominal* — labels with no order (Sex, Embarked port).
- *Ordinal* — categories with a natural order (Pclass: 1st > 2nd > 3rd).
- *Binary* — exactly two values (Survived: 0 or 1).

In [ ]:
# Classifying every column of our dataset:
#   Survived → qualitative, binary        (0/1)
#   Pclass   → qualitative, ordinal        (1 > 2 > 3)
#   Sex      → qualitative, nominal        (male/female)
#   Age      → quantitative, continuous
#   Fare     → quantitative, continuous
#   Embarked → qualitative, nominal        (S/C/Q)
df.dtypes

Note that the *stored* dtype (`int64`, `object`) is not the same as the *conceptual* type. `Pclass` is stored as an integer but is really an ordinal category — the number is a rank, not a quantity. Judgement, not the dtype, tells you how to treat a column.

### `value_counts()` — counting categories

For qualitative columns, `value_counts()` tallies how often each category appears. The top row is the **mode** (the most frequent value).

In [ ]:
df["Embarked"].value_counts()      # the most common port is the mode of this column

In [ ]:
df["Pclass"].value_counts()

---
## 7. Everyday DataFrame operations

In [ ]:
df["Sex"].unique()        # the distinct values in a column

In [ ]:
df["Pclass"].nunique()    # HOW MANY distinct values

`apply()` runs a function on every value of a column — the same idea as the lambda functions from the Python weeks.

In [ ]:
# Turn Fare into a simple category with a lambda
df["fare_level"] = df["Fare"].apply(lambda x: "high" if x > 30 else "low")
df[["Fare", "fare_level"]].head()

In [ ]:
# Sorting by a column
df.sort_values("Fare", ascending=False).head()

In [ ]:
# Renaming columns
df.rename(columns={"Pclass": "PassengerClass"}).head(3)

In [ ]:
# Dropping a column (axis=1 means "a column"; axis=0 would mean "a row")
df.drop("fare_level", axis=1).head(3)

### Checking for missing data

Real datasets have gaps. `isnull()` marks each cell as missing (True) or present (False); summing gives a per-column count of what's missing.

In [ ]:
df.isnull().sum()      # how many values are missing in each column

`Age` has missing values, but `Fare`, `Sex`, and the rest are complete. Deciding what to do with those gaps — remove the rows, or fill them in — is the first task of the next session, along with grouping, merging, and correlation.

---
## Summary

- A **Series** is a labeled 1-D array; a **DataFrame** is a table of Series.
- `pd.read_csv()` loads data; `.head()`, `.info()`, `.shape`, `.dtypes` inspect it.
- Select columns with `df["col"]`, rows with `.loc` (labels) and `.iloc` (positions), and subsets with boolean masks.
- `.describe()` reports mean, std, min, max, and quartiles for every numeric column at once.
- Classify each column as quantitative (continuous/discrete) or qualitative (nominal/ordinal/binary) before analysing it.
- `value_counts()`, `unique()`, `apply()`, `sort_values()`, and `isnull().sum()` are the everyday tools you will reach for constantly.